# **Tahap 3 - Load ke SQL Database**

## Import Library

In [1]:
import shutil
import sqlite3
import tempfile
import pandas as pd
from pathlib import Path

## Path Konfigurasi

In [2]:
BASE_DIR      = Path(__file__).resolve().parent.parent \
                if "__file__" in dir() else Path.cwd().parent
 
CLEANED_DIR   = BASE_DIR / "data" / "cleaned"
DATABASE_DIR  = BASE_DIR / "database"
 
if not CLEANED_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/cleaned tidak ditemukan: {CLEANED_DIR}\n"
        f"Jalankan dulu: python src/02_cleaning.py"
    )
 
DATABASE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DATABASE_DIR / "crime_indonesia.db"

## Mapping Provinsi -> Pulau dan Region

In [3]:
PROVINCE_GEO = {
    "Aceh"                     : ("Sumatera",  "Sumatera Bagian Utara"),
    "Sumatera Utara"           : ("Sumatera",  "Sumatera Bagian Utara"),
    "Sumatera Barat"           : ("Sumatera",  "Sumatera Bagian Tengah"),
    "Riau"                     : ("Sumatera",  "Sumatera Bagian Tengah"),
    "Jambi"                    : ("Sumatera",  "Sumatera Bagian Tengah"),
    "Sumatera Selatan"         : ("Sumatera",  "Sumatera Bagian Selatan"),
    "Bengkulu"                 : ("Sumatera",  "Sumatera Bagian Selatan"),
    "Lampung"                  : ("Sumatera",  "Sumatera Bagian Selatan"),
    "Kepulauan Bangka Belitung": ("Sumatera",  "Sumatera Bagian Selatan"),
    "Kepulauan Riau"           : ("Sumatera",  "Sumatera Bagian Tengah"),
    "DKI Jakarta"              : ("Jawa",      "Jawa Bagian Barat"),
    "Jawa Barat"               : ("Jawa",      "Jawa Bagian Barat"),
    "Banten"                   : ("Jawa",      "Jawa Bagian Barat"),
    "Jawa Tengah"              : ("Jawa",      "Jawa Bagian Tengah"),
    "DI Yogyakarta"            : ("Jawa",      "Jawa Bagian Tengah"),
    "Jawa Timur"               : ("Jawa",      "Jawa Bagian Timur"),
    "Bali"                     : ("Bali Nusra","Bali"),
    "Nusa Tenggara Barat"      : ("Bali Nusra","Nusa Tenggara"),
    "Nusa Tenggara Timur"      : ("Bali Nusra","Nusa Tenggara"),
    "Kalimantan Barat"         : ("Kalimantan","Kalimantan Bagian Barat"),
    "Kalimantan Tengah"        : ("Kalimantan","Kalimantan Bagian Tengah"),
    "Kalimantan Selatan"       : ("Kalimantan","Kalimantan Bagian Selatan"),
    "Kalimantan Timur"         : ("Kalimantan","Kalimantan Bagian Timur"),
    "Kalimantan Utara"         : ("Kalimantan","Kalimantan Bagian Timur"),
    "Sulawesi Utara"           : ("Sulawesi",  "Sulawesi Bagian Utara"),
    "Gorontalo"                : ("Sulawesi",  "Sulawesi Bagian Utara"),
    "Sulawesi Tengah"          : ("Sulawesi",  "Sulawesi Bagian Tengah"),
    "Sulawesi Barat"           : ("Sulawesi",  "Sulawesi Bagian Barat"),
    "Sulawesi Selatan"         : ("Sulawesi",  "Sulawesi Bagian Selatan"),
    "Sulawesi Tenggara"        : ("Sulawesi",  "Sulawesi Bagian Selatan"),
    "Maluku"                   : ("Maluku Papua","Maluku"),
    "Maluku Utara"             : ("Maluku Papua","Maluku"),
    "Papua Barat"              : ("Maluku Papua","Papua"),
    "Papua"                    : ("Maluku Papua","Papua"),
}

## Data Definition Language - CREATE TABLES

In [4]:
DDL = """
-- ============================================================
-- DIMENSI
-- ============================================================
 
CREATE TABLE IF NOT EXISTS dim_province (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    provinsi TEXT    NOT NULL UNIQUE,
    pulau    TEXT,
    region   TEXT
);
 
CREATE TABLE IF NOT EXISTS dim_year (
    tahun INTEGER PRIMARY KEY
);
 
-- ============================================================
-- FAKTA
-- ============================================================
 
CREATE TABLE IF NOT EXISTS fact_crime (
    id                 INTEGER PRIMARY KEY AUTOINCREMENT,
    province_id        INTEGER NOT NULL REFERENCES dim_province(id),
    tahun              INTEGER NOT NULL REFERENCES dim_year(tahun),
    crime_total        INTEGER,
    is_outlier         INTEGER DEFAULT 0,   -- 1 = outlier, nilai crime_total = NULL
    crime_rate_per100k REAL,
    UNIQUE (province_id, tahun)
);
 
CREATE TABLE IF NOT EXISTS fact_population (
    id                 INTEGER PRIMARY KEY AUTOINCREMENT,
    province_id        INTEGER NOT NULL REFERENCES dim_province(id),
    tahun              INTEGER NOT NULL REFERENCES dim_year(tahun),
    population         INTEGER,
    population_density REAL,
    is_extrapolated    INTEGER DEFAULT 0,   -- 1 = dihitung dari ekstrapolasi mundur
    UNIQUE (province_id, tahun)
);
 
CREATE TABLE IF NOT EXISTS fact_unemployment (
    id                   INTEGER PRIMARY KEY AUTOINCREMENT,
    province_id          INTEGER NOT NULL REFERENCES dim_province(id),
    tahun                INTEGER NOT NULL REFERENCES dim_year(tahun),
    unemployment_rate    REAL,    -- rata-rata Feb + Agustus
    UNIQUE (province_id, tahun)
);
 
-- ============================================================
-- INDEX
-- ============================================================
 
CREATE INDEX IF NOT EXISTS idx_crime_tahun      ON fact_crime(tahun);
CREATE INDEX IF NOT EXISTS idx_crime_province   ON fact_crime(province_id);
CREATE INDEX IF NOT EXISTS idx_pop_tahun        ON fact_population(tahun);
CREATE INDEX IF NOT EXISTS idx_unemp_tahun      ON fact_unemployment(tahun);
"""

## Data Definition Language - CREATE VIEWS

In [5]:
VIEWS = """
-- ============================================================
-- VIEW: Ringkasan lengkap per provinsi per tahun
-- ============================================================
CREATE VIEW IF NOT EXISTS v_crime_summary AS
SELECT
    p.provinsi,
    p.pulau,
    p.region,
    c.tahun,
    c.crime_total,
    c.is_outlier,
    c.crime_rate_per100k,
    pop.population,
    pop.population_density,
    pop.is_extrapolated  AS pop_extrapolated,
    u.unemployment_rate
FROM fact_crime c
JOIN dim_province p   ON p.id    = c.province_id
LEFT JOIN fact_population   pop ON pop.province_id = c.province_id AND pop.tahun = c.tahun
LEFT JOIN fact_unemployment u   ON u.province_id   = c.province_id AND u.tahun   = c.tahun;
 
-- ============================================================
-- VIEW: Top 10 provinsi paling berbahaya per tahun
-- ============================================================
CREATE VIEW IF NOT EXISTS v_top_dangerous AS
SELECT
    tahun,
    provinsi,
    crime_rate_per100k,
    crime_total,
    population,
    ROW_NUMBER() OVER (PARTITION BY tahun ORDER BY crime_rate_per100k DESC) AS ranking
FROM v_crime_summary
WHERE crime_rate_per100k IS NOT NULL
  AND is_outlier = 0;
 
-- ============================================================
-- VIEW: Tren nasional per tahun
-- ============================================================
CREATE VIEW IF NOT EXISTS v_trend_national AS
SELECT
    tahun,
    SUM(crime_total)                        AS total_crime_nasional,
    ROUND(AVG(crime_rate_per100k), 1)       AS avg_crime_rate,
    MAX(crime_rate_per100k)                 AS max_crime_rate,
    MIN(crime_rate_per100k)                 AS min_crime_rate,
    COUNT(DISTINCT provinsi)                AS jumlah_provinsi
FROM v_crime_summary
WHERE is_outlier = 0
  AND crime_total IS NOT NULL
GROUP BY tahun
ORDER BY tahun;
"""
 

## Function 

In [ ]:
def create_schema(conn: sqlite3.Connection) -> None:
    """Buat semua tabel dan index."""
    conn.executescript(DDL)
    conn.commit()
    print("  ✅ Tabel & index dibuat")
 
 
def create_views(conn: sqlite3.Connection) -> None:
    """Buat semua views."""
    conn.executescript(VIEWS)
    conn.commit()
    print("Views dibuat")
 
 
def load_dim_province(conn: sqlite3.Connection, provinces: list) -> dict:
    """
    Insert provinsi ke dim_province.
    Return dict: {nama_provinsi: id}
    """
    cur = conn.cursor()
    for prov in sorted(provinces):
        pulau, region = PROVINCE_GEO.get(prov, (None, None))
        cur.execute(
            "INSERT OR IGNORE INTO dim_province (provinsi, pulau, region) VALUES (?, ?, ?)",
            (prov, pulau, region)
        )
    conn.commit()
 
    rows = cur.execute("SELECT id, provinsi FROM dim_province").fetchall()
    prov_map = {row[1]: row[0] for row in rows}
    print(f"dim_province → {len(prov_map)} provinsi")
    return prov_map
 
 
def load_dim_year(conn: sqlite3.Connection, years: list) -> None:
    """Insert tahun ke dim_year."""
    cur = conn.cursor()
    for yr in years:
        cur.execute("INSERT OR IGNORE INTO dim_year (tahun) VALUES (?)", (int(yr),))
    conn.commit()
    print(f"dim_year → tahun {min(years)}–{max(years)}")
 
 
def load_fact_crime(conn: sqlite3.Connection, df: pd.DataFrame, prov_map: dict) -> None:
    """Insert data crime ke fact_crime."""
    cur  = conn.cursor()
    rows = 0
    for _, r in df.iterrows():
        pid = prov_map.get(r['provinsi'])
        if pid is None:
            continue
        cur.execute("""
            INSERT OR REPLACE INTO fact_crime
                (province_id, tahun, crime_total, is_outlier, crime_rate_per100k)
            VALUES (?, ?, ?, ?, ?)
        """, (
            pid,
            int(r['tahun']),
            None if pd.isna(r.get('crime_total')) else int(r['crime_total']),
            int(r.get('is_outlier', 0)),
            None if pd.isna(r.get('crime_rate_per100k')) else float(r['crime_rate_per100k']),
        ))
        rows += 1
    conn.commit()
    print(f"fact_crime → {rows} rows")
 
 
def load_fact_population(conn: sqlite3.Connection, df: pd.DataFrame, prov_map: dict) -> None:
    """Insert data populasi ke fact_population."""
    cur  = conn.cursor()
    rows = 0
    for _, r in df.iterrows():
        pid = prov_map.get(r['provinsi'])
        if pid is None:
            continue
        cur.execute("""
            INSERT OR REPLACE INTO fact_population
                (province_id, tahun, population, population_density, is_extrapolated)
            VALUES (?, ?, ?, ?, ?)
        """, (
            pid,
            int(r['tahun']),
            None if pd.isna(r.get('population'))         else int(r['population']),
            None if pd.isna(r.get('population_density')) else float(r['population_density']),
            int(r.get('is_extrapolated', 0)),
        ))
        rows += 1
    conn.commit()
    print(f"fact_population → {rows} rows")
 
 
def load_fact_unemployment(conn: sqlite3.Connection, df: pd.DataFrame, prov_map: dict) -> None:
    """Insert data pengangguran ke fact_unemployment."""
    cur  = conn.cursor()
    rows = 0
    rate_col = next(
        (c for c in ['unemployment_rate_avg', 'unemployment_rate'] if c in df.columns),
        None
    )
    if rate_col is None:
        print("Kolom unemployment_rate tidak ditemukan, skip")
        return
 
    for _, r in df.iterrows():
        pid = prov_map.get(r['provinsi'])
        if pid is None:
            continue
        cur.execute("""
            INSERT OR REPLACE INTO fact_unemployment
                (province_id, tahun, unemployment_rate)
            VALUES (?, ?, ?)
        """, (
            pid,
            int(r['tahun']),
            None if pd.isna(r[rate_col]) else float(r[rate_col]),
        ))
        rows += 1
    conn.commit()
    print(f"fact_unemployment → {rows} rows")
 
 
def verify_database(conn: sqlite3.Connection) -> None:
    """Jalankan beberapa query verifikasi."""
    print("\n  --- Verifikasi Query ---")
    cur = conn.cursor()
 
    checks = [
        ("Jumlah provinsi"   , "SELECT COUNT(*) FROM dim_province"),
        ("Tahun tersedia"    , "SELECT MIN(tahun)||'–'||MAX(tahun) FROM dim_year"),
        ("Total crime rows"  , "SELECT COUNT(*) FROM fact_crime"),
        ("Total pop rows"    , "SELECT COUNT(*) FROM fact_population"),
        ("Total unemp rows"  , "SELECT COUNT(*) FROM fact_unemployment"),
    ]
    for label, query in checks:
        result = cur.execute(query).fetchone()[0]
        print(f"{label:<22}: {result}")
 
    print("\n  Top 3 berbahaya 2024 (via view):")
    rows = cur.execute("""
        SELECT provinsi, crime_rate_per100k, ranking
        FROM   v_top_dangerous
        WHERE  tahun = 2024 AND ranking <= 3
    """).fetchall()
    for row in rows:
        print(f"    #{int(row[2])} {row[0]:<30} {row[1]} / 100k")
 
    print("\n  Tren nasional (via view):")
    rows = cur.execute("""
        SELECT tahun, total_crime_nasional, avg_crime_rate
        FROM   v_trend_national
        ORDER  BY tahun
    """).fetchall()
    for row in rows:
        print(f"    {row[0]}: total={row[1]:>7,}  avg rate={row[2]}")
 
 
def export_schema_sql(conn: sqlite3.Connection) -> None:
    """Export skema ke file schema.sql untuk dokumentasi."""
    schema_path = DATABASE_DIR / "schema.sql"
    lines = ["-- Auto-generated schema\n\n"]
    cur = conn.cursor()
 
    cur.execute("SELECT sql FROM sqlite_master WHERE type='table' ORDER BY name")
    for row in cur.fetchall():
        if row[0]:
            lines.append(row[0] + ";\n\n")
 
    cur.execute("SELECT sql FROM sqlite_master WHERE type='view' ORDER BY name")
    for row in cur.fetchall():
        if row[0]:
            lines.append(row[0] + ";\n\n")
 
    schema_path.write_text("".join(lines), encoding="utf-8")
    print(f"\nSchema exported → {schema_path}")

## Main

In [7]:
def main():
    print("\n" + "=" * 60)
    print("TAHAP 3 — LOAD KE SQL DATABASE")
    print(f"Database : {DB_PATH}")
    print("=" * 60)

    df_merged = pd.read_csv(CLEANED_DIR / "crime_merged.csv")
    df_pop    = pd.read_csv(CLEANED_DIR / "population_cleaned.csv")
    df_unemp  = pd.read_csv(CLEANED_DIR / "unemployment_cleaned.csv")
 
    print(f"\n  Loaded crime_merged.csv    : {len(df_merged)} rows")
    print(f"  Loaded population_cleaned  : {len(df_pop)} rows")
    print(f"  Loaded unemployment_cleaned: {len(df_unemp)} rows")

    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as tmp:
        tmp_path = Path(tmp.name)
 
    with sqlite3.connect(tmp_path) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.execute("PRAGMA journal_mode = WAL")
 
        print("\n--- Buat Skema ---")
        create_schema(conn)
 
        print("\n--- Load Dimensi ---")
        all_provinces = sorted(df_merged['provinsi'].unique())
        all_years     = sorted(
            set(df_merged['tahun'].unique()) | set(df_pop['tahun'].unique())
        )
        prov_map = load_dim_province(conn, all_provinces)
        load_dim_year(conn, all_years)
 
        print("\n--- Load Fakta ---")
        load_fact_crime(conn, df_merged, prov_map)
        load_fact_population(conn, df_pop, prov_map)
        load_fact_unemployment(conn, df_unemp, prov_map)
 
        print("\n--- Buat Views ---")
        create_views(conn)
 
        print("\n--- Verifikasi ---")
        verify_database(conn)
 
        export_schema_sql(conn)

    shutil.copy2(tmp_path, DB_PATH)
    tmp_path.unlink(missing_ok=True)
 
    size_kb = DB_PATH.stat().st_size / 1024
    print(f"\n{'=' * 60}")
    print(f" Database selesai: {DB_PATH}")
    print(f"  Ukuran: {size_kb:.1f} KB")
    print(f"\nTahap 3 selesai. Lanjut ke: src/04_preprocessing.py")
 
 
if __name__ == "__main__":
    main()


TAHAP 3 — LOAD KE SQL DATABASE
Database : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/database/crime_indonesia.db

  Loaded crime_merged.csv    : 331 rows
  Loaded population_cleaned  : 381 rows
  Loaded unemployment_cleaned: 102 rows

--- Buat Skema ---
  ✅ Tabel & index dibuat

--- Load Dimensi ---
  ✅ dim_province → 34 provinsi
  ✅ dim_year → tahun 2015–2026

--- Load Fakta ---
  ✅ fact_crime → 331 rows
  ✅ fact_population → 381 rows
  ✅ fact_unemployment → 99 rows

--- Buat Views ---
  ✅ Views dibuat

--- Verifikasi ---

  --- Verifikasi Query ---
  ✅ Jumlah provinsi       : 34
  ✅ Tahun tersedia        : 2015–2026
  ✅ Total crime rows      : 331
  ✅ Total pop rows        : 381
  ✅ Total unemp rows      : 99

  Top 3 berbahaya 2024 (via view):
    #1 Papua Barat                    1051.5 / 100k
    #2 DKI Jakarta                    723.1 / 100k
    #3 Sulawesi Utara                 484.0 / 100k

  Tren nasional (via view):
    2015: total=317,532  avg rate=189.3
   